In [14]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import sklearn
sklearn.set_config(display='text')
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

from sklearn.linear_model import LinearRegression # 일반 선형 회귀 모델을 사용하기 위해 import 한다.
from sklearn.linear_model import Lasso # 라쏘 선형 회귀 모델을 사용하기 위해 import 한다.
from sklearn.linear_model import Ridge # 릿지 선형 회귀 모델을 사용하기 위해 import 한다.
from sklearn.linear_model import ElasticNet # 엘라스틱넷 선형 회귀 모델을 사용하기 위해 import 한다.

선형 회귀 분석은 가장 기본적인 머신러닝 방법이며, 여러 분야에서 사용된다.

선형 회귀 분석은 피쳐와 레이블 간의 선형 관계를 파악하는 알고리즘으로 피쳐 데이터($x$)와 피쳐 데이터에 따른 레이블($y$) 사이의 선형 관계가 존재할 때 이를 수식화 하면 다음과 같다.

$$y=wx+b=ax+b$$

데이터 $x$가 주어질 때, 데이터 $x$에 가중치(weight) $w$를 곱하고 바이어스 $b$를 더하면 레이블을 얻을 수 있다. 선형 회귀에서 해야할 일은 피쳐 $x$와 레이블 $y$를 이용해서 최적의 가중치 $w$와 바이어스 $b$를 구하는 것이다.

위의 수식은 피쳐의 개수가 1개일 경우이고, 이를 일반화시켜 $p$개의 피쳐를 가지는 데이터라고 가정하면 데이터셋을 구성하는 각 데이터의 포인트는 $p$개의 학습 데이터로 구성되므로 $x=(x_1, x_2, x_3, ..., x_p)$라고 표현할 수 있다. $x_i$는 i번째 데이터를 열 벡터로 표현한 것이다. 그래서 선형 회귀 모델은 아래와 같다.

$$y = \hat y = w_1x_1 + w_2x_2 + w_3x_3 + ... + w_px_p$$

이때, $w = (w_1, w_2, w_3, ..., w_p)^T$를 가중치라 부른다. 각 가중치 요소 하나하나가 우리가 구하려는 파라미터이며, 파라미터 값은 예측에 영향을 미친다. 즉, 파라미터 값에 따라서 예측이 달라진다.

가중치는 학습 데이터로 부터 최소 제곱법(least squared etimator)를 사용해 구하 수 있다. 최소 제곱법은 오차의 제곱합이 최소가 되는 추정량을 구하는 방법이다.

캘리포이아 집값 데이터를 사용해서 캘리포니아 집값을 예측하는 모델을 생성하고 학습시킨다.

In [17]:
# 데이터 불러오기
raw_data = datasets.fetch_california_housing() # 사이킷런 라이브러리가 제공하는 캘리포니아 집값 데이터를 불러온다.
# print(raw_data)

# 피쳐, 레이블 데이터 저장
xData = raw_data.data # 피쳐 데이터를 저장한다.
yData = raw_data.target # 피쳐 데이터에 따른 레이블을 저장한다.
# print(xData.shape, yData.shape)

# 학습 데이터와 테스트 데이터로 분할
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

# 데이터 표준화(정규화)
scaler = StandardScaler() # 표준화 스케일러 객체를 만든다.
x_train = scaler.fit_transform(x_train) # 학습 데이터를 표준화 스케일러로 표준화하고 적용한다.
x_test = scaler.transform(x_test) # 테스트 데이터를 학습 데이터로 표준화한 스케일러에 적용한다.

# 모델 생성 후 데이터 학습
# model_linear = LinearRegression() # 일반 선형 회귀 모델을 만든다.
# model_linear.fit(x_train, y_train) # 표준화된 학습 데이터(x_train)와 학습 데이터에 따른 레이블(y_train)을 넘겨서 선형 회귀 모델을 학습시킨다.
model_linear = LinearRegression().fit(x_train, y_train) # 선형 회귀 모델을 만들고 학습시킨다.

In [3]:
# 일반 선형 회귀 계수(가중치)와 상수항(바이어스)
print(model_linear.coef_) # 회귀 계수(가중치)
print(model_linear.intercept_) # 상수항(바이어스)

[ 0.83189945  0.1209374  -0.26175157  0.30405212 -0.00873559 -0.02984442
 -0.89236538 -0.86385031]
2.074372627906947


라쏘(lasso) 선형 회귀 분석(L1 제약식 사용)

라쏘를 사용하는 이유는 '불필요한 변수를 자동으로 제거하여 모델을 단순하고 해석하기 쉽게 만들기 위해서'이다.

L1 제약식은 머신러닝, 특히 선형 회귀 모델에서 과적합을 막고 일반화 성능을 높이기 위해 사용하는 가중치 규제 기법으로 모델이 학습할 때 가중치(회귀 계수)가 너무 커지지 않도록 제약을 거는 방식이며, 수학적으로는 가중치들의 절대값의 합을 제한 조건으로 건다.

L1 제약식의 특징은 중요도가 떨어지거나 불필요한 피쳐의 가중치를 0으로 만든다.

In [4]:
# 라쏘 선형 회귀 모델의 alpha 속성으로 제약 정도를 지정해서 모델을 만든다.
# 라쏘 선형 회귀 모델에서 alpha는 규제의 강도를 조절하는 하이퍼파라미터로 alpha 속성의 기본값은 1이고 값이 클 수록 강한 제약을 의미한다.
# alpha 속성의 수학적 범위는 0 이상 무한대 까지이고 실무적 범위는 로그 스케일(0.001, 0.01, 0.1, 1, 10, 100, ...)값을 사용한다.
# alpha 속성의 값이 0이면 규제가 전혀 적용되지 않아서 일반 선형 회귀와 똑같이 동작한다.
# model_lasso = Lasso(alpha=0.01)
# model_lasso.fit(x_train, y_train)
model_lasso = Lasso(alpha=0.01).fit(x_train, y_train)

In [5]:
# 라쏘 선형 회귀 계수(가중치)와 상수항(바이어스)
print(model_lasso.coef_) # 회귀 계수(가중치)
print(model_lasso.intercept_) # 상수항(바이어스)

[ 0.77897391  0.12866563 -0.11947889  0.16185133 -0.         -0.02249526
 -0.78774759 -0.75273912]
2.0743726279069525


릿지(ridge) 선형 회귀 분석(L2 제약식 사용)

L2 제약식은 머신러닝, 특히 선형 회귀 모델에서 과적합을 막고 일반화 성능을 높이기 위해 사용하는 가중치 규제 기법으로 모델이 학습할 때 가중치(회귀 계수)가 너무 커지지 않도록 제약을 거는 방식이며, 수학적으로는 가중치들의 제곱의 합을 제한 조건으로 건다.

L2 제약식의 특징은 L1 제약식과 마찬가지로 중요도가 떨어지거나 불필요한 피쳐의 가중치를 0으로 만든다.

In [6]:
# 릿지 선형 회귀 모델의 alpha 속성으로 제약 정도를 지정해서 모델을 만든다. alpha 속성의 사용 방법은 릿지 선형 회귀 모델과 같다.
# model_ridge = Ridge(alpha=1)
# model_ridge.fit(x_train, y_train)
model_ridge = Ridge(alpha=1).fit(x_train, y_train)

In [7]:
# 릿지 선형 회귀 계수(가중치)와 상수항(바이어스)
print(model_ridge.coef_) # 회귀 계수(가중치)
print(model_ridge.intercept_) # 상수항(바이어스)

[ 0.83186352  0.12102542 -0.26157085  0.30381865 -0.00870202 -0.02985494
 -0.89154458 -0.86302194]
2.074372627906947


엘라스틱넷(elastic net) 선형 회귀 분석(L1, L2 제약식 모두 사용)

엘라스틱넷은 라쏘(Lasso, L1 규제)와 릿지(Ridge, L2 규제)의 장점을 결합한 하이브리드 선형 회귀 규제 모델로 라쏘와 릿지가 각각 가진 한계점을 보완하기 위해 만들어졌으며, 특히 변수 간 상관관계가 높고 데이터가 복잡할 때 사용한다.

In [8]:
# 엘라스틱넷 선형 회귀 모델의 alpha 속성으로 제약 정도를 지정해서 모델을 만든다.
# alpha 속성에는 L1 제약식의 크기와 L2 제약식의 크기의 합을 지정하고 l1_ratio 속성으로 alpha 속성에 지정한 값에서 L1 제약이 차지하는 비율을 지정한다.
# l1_ratio 속성값은 비율이므로 0부터 1 사이의 값을 지정해야 하며, 0을 지정하면 L1 제약식이 사용되지 않고 L2 제약만 사용하는 릿지 선형 회귀 분석을 의미하고,
# 1을 지정하면 L2 제약식이 사용되지 않고 L1 제약만 사용하는 라쏘 선형 회귀 분석이 된다.
# model_elastic = ElasticNet(alpha=0.01, l1_ratio=0.01)
# model_elastic.fit(x_train, y_train)
model_elastic = ElasticNet(alpha=0.01, l1_ratio=0.01).fit(x_train, y_train)

In [9]:
# 엘라스틱넷 선형 회귀 계수(가중치)와 상수항(바이어스)
print(model_elastic.coef_) # 회귀 계수(가중치)
print(model_elastic.intercept_) # 상수항(바이어스)

[ 0.8239746   0.13234248 -0.23241126  0.26811823 -0.00424703 -0.03112002
 -0.78337789 -0.75358665]
2.074372627906953


학습된 모델로 테스트 데이터를 예측한다.

In [10]:
# predict() 함수의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 선형 회귀 모델(일반, 라쏘, 릿지, 엘라스틱넷)을 예측한다.
predict_linear = model_linear.predict(x_test)
predict_lasso = model_lasso.predict(x_test)
predict_ridge = model_ridge.predict(x_test)
predict_elastic = model_elastic.predict(x_test)

for i in range(len(predict_linear))[:5]:
    print('일반: {:8.5f}, 라쏘: {:8.5f}, 릿지: {:8.5f}, 엘라: {:8.5f}'.format(predict_linear[i], predict_lasso[i], predict_ridge[i], predict_elastic[i]))

일반:  2.27826, 라쏘:  2.27774, 릿지:  2.27811, 엘라:  2.25850
일반:  2.79607, 라쏘:  2.83514, 릿지:  2.79618, 엘라:  2.81142
일반:  1.90887, 라쏘:  1.96669, 릿지:  1.90916, 엘라:  1.94769
일반:  1.02576, 라쏘:  1.07487, 릿지:  1.02638, 엘라:  1.10835
일반:  2.95860, 라쏘:  2.81830, 릿지:  2.95813, 엘라:  2.89183


학습된 모델을 평가한다.

In [11]:
# 평균 절대값 오차(MAE)
# mean_absolute_error() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 평균 절대값 오차를 계산한다.
print('일반:', mean_absolute_error(y_test, predict_linear))
print('라쏘:', mean_absolute_error(y_test, predict_lasso))
print('릿지:', mean_absolute_error(y_test, predict_ridge))
print('엘라:', mean_absolute_error(y_test, predict_elastic))

일반: 0.5368950735045218
라쏘: 0.5409381699751551
릿지: 0.5368952963837788
엘라: 0.5380423238460021


In [12]:
# 평균 제곱 오차(MSE)
# mean_squared_error() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 평균 제곱 오차를 계산한다.
print('일반:', mean_squared_error(y_test, predict_linear))
print('라쏘:', mean_squared_error(y_test, predict_lasso))
print('릿지:', mean_squared_error(y_test, predict_ridge))
print('엘라:', mean_squared_error(y_test, predict_elastic))

일반: 0.5404128061709079
라쏘: 0.547654376568303
릿지: 0.5404220132440146
엘라: 0.5435117084883777


In [13]:
# 평균 제곱근 오차(RMSE)
# mean_squared_error() 함수의 실행 결과를 루트를 씌우면 RMSE가 된다.
print('일반:', np.sqrt(mean_squared_error(y_test, predict_linear)))
print('라쏘:', np.sqrt(mean_squared_error(y_test, predict_lasso)))
print('릿지:', np.sqrt(mean_squared_error(y_test, predict_ridge)))
print('엘라:', np.sqrt(mean_squared_error(y_test, predict_elastic)))

일반: 0.7351277481981672
라쏘: 0.7400367400124828
릿지: 0.7351340103981142
엘라: 0.7372324657042565


In [15]:
# R2 score
# r2_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 R2 score를 계산한다.
print('일반:', r2_score(y_test, predict_linear))
print('라쏘:', r2_score(y_test, predict_lasso))
print('릿지:', r2_score(y_test, predict_ridge))
print('엘라:', r2_score(y_test, predict_elastic))

일반: 0.5911695436410489
라쏘: 0.5856911861770583
릿지: 0.5911625783510761
엘라: 0.588825177197108
